# Pre-RAG Demo

This notebook shows how a plain LLM answers questions before document retrieval is added.

The model answers from general knowledge only, so private document facts and project-specific numbers may be missing or unverified.

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm_provider = os.getenv("LLM_PROVIDER", "openai").lower()

if llm_provider == "ollama":
    from langchain_ollama import ChatOllama

    plain_llm = ChatOllama(
        model=os.getenv("OLLAMA_MODEL", "llama3.2:latest"),
        base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
        temperature=0,
    )
else:
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Missing OPENAI_API_KEY. Add it to your .env file.")

    plain_llm = ChatOpenAI(
        model=os.getenv("OPENAI_LLM_MODEL", "gpt-4o"),
        temperature=0,
        openai_api_key=os.getenv("OPENAI_API_KEY"),
    )

print(f"Pre-RAG plain LLM ready: {llm_provider}")

c:\Users\Saurav Kumar\anaconda3\envs\tf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pre-RAG plain LLM ready: openai


In [1]:
# Pick questions that normally need your private documents or knowledge base.
# These help show the limitation of a pre-RAG plain LLM.
questions = [
    "What ROI do companies report from RAG deployments?",
    "What percentage of Fortune 500 companies use RAG in 2026?",
    "What are the main obstacles to enterprise RAG adoption?",
    "what is difference between lora and Qlora?",
]

questions

['What ROI do companies report from RAG deployments?',
 'What percentage of Fortune 500 companies use RAG in 2026?',
 'What are the main obstacles to enterprise RAG adoption?',
 'what is difference between lora and Qlora?']

In [4]:
def response_text(response):
    """Normalize LangChain model responses to plain text."""
    return getattr(response, "content", str(response))


def ask_pre_rag(question: str) -> str:
    prompt = f"""
You are answering before document retrieval is added.
You do not have access to the user's private documents or retrieved context.
Answer from general knowledge only.
If the question depends on private or project-specific data, say that you cannot verify it without those documents.

Question: {question}
""".strip()
    return response_text(plain_llm.invoke(prompt))


pre_rag_answers = {question: ask_pre_rag(question) for question in questions}

for question, answer in pre_rag_answers.items():
    print("=" * 80)
    print("Pre-RAG")
    print(f"Question: {question}\n")
    print(answer)
    print()

Pre-RAG
Question: What ROI do companies report from RAG deployments?

I cannot provide specific ROI figures from RAG (Retrieval-Augmented Generation) deployments without access to private or project-specific data. However, generally speaking, companies that implement RAG systems often report improvements in efficiency, accuracy, and user satisfaction. These systems can enhance information retrieval and decision-making processes by combining the strengths of retrieval-based and generative models. The ROI can vary significantly depending on the specific application, industry, and how well the system is integrated into existing workflows. For precise ROI figures, companies typically conduct internal assessments based on their unique use cases and performance metrics.

Pre-RAG
Question: What percentage of Fortune 500 companies use RAG in 2026?

I cannot provide specific information about the percentage of Fortune 500 companies using Retrieval-Augmented Generation (RAG) in 2026, as this wou

## What This Shows

This is the plain LLM baseline. The model can answer only from its general training, so project-specific numbers, private document facts, and current internal details may be missing or unverified.